In [1]:
# Run this cell first. It installs dependencies into the active notebook kernel.
import subprocess
import sys

subprocess.check_call([sys.executable, "-m", "pip", "install", "--upgrade", "pip"])
packages = [
    "pandas>=2.2,<3",
    "numpy>=1.26,<3",
    "scikit-learn>=1.4,<2",
    "matplotlib>=3.7,<4",
    "seaborn>=0.13,<1",
    "joblib>=1.3,<2",
    "ipykernel",
]
subprocess.check_call([sys.executable, "-m", "pip", "install", "--upgrade", *packages])

print("Dependencies installed. Restart the kernel and run all remaining cells if imports previously failed.")


Dependencies installed. Restart the kernel and run all remaining cells if imports previously failed.


# NYC Taxi Gold Demand Analysis

This notebook uses only the four exported gold CSV files in `data/`:
`dim_taxi_zone.csv`, `fact_pickups_hourly.csv`, `ml_training_features.csv`, and
`ml_inference_features.csv`.

It trains a histogram gradient-boosting forecaster, evaluates it with a chronological
split, scores the inference data, detects zone-specific residual anomalies, and
segments zones into four demand profiles with K-Means.

In [2]:
from pathlib import Path

import joblib
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import seaborn as sns
from sklearn.cluster import KMeans
from sklearn.ensemble import HistGradientBoostingRegressor
from sklearn.inspection import permutation_importance
from sklearn.metrics import mean_absolute_error, mean_squared_error, r2_score
from sklearn.preprocessing import StandardScaler

plt.style.use("seaborn-v0_8-whitegrid")
sns.set_context("notebook")

candidates = [Path.cwd(), *Path.cwd().parents]
project_dir = next((candidate for candidate in candidates if (candidate / "data").is_dir()), None)
if project_dir is None:
    raise FileNotFoundError("Could not locate the repository folder containing data/.")
data_dir = project_dir / "data"

zone = pd.read_csv(data_dir / "dim_taxi_zone.csv", parse_dates=["processed_at"])
fact = pd.read_csv(data_dir / "fact_pickups_hourly.csv", parse_dates=["pickup_hour", "processed_at"])
train = pd.read_csv(data_dir / "ml_training_features.csv", parse_dates=["forecast_hour", "processed_at"])
inference = pd.read_csv(data_dir / "ml_inference_features.csv", parse_dates=["forecast_hour", "processed_at"])

print({
    "zones": len(zone),
    "hourly_fact_rows": len(fact),
    "training_rows": len(train),
    "inference_rows": len(inference),
})
print("Coverage:", fact["pickup_hour"].min(), "to", fact["pickup_hour"].max())

assert len(zone) == zone["pickup_zone_id"].nunique()
assert not fact.duplicated(["pickup_zone_id", "pickup_hour"]).any()
assert not train.duplicated(["pickup_zone_id", "forecast_hour"]).any()
assert not inference.duplicated(["pickup_zone_id", "forecast_hour"]).any()

excluded = {"pickup_zone_id", "forecast_hour", "processed_at", "target_pickup_count"}
train_features = [column for column in train.columns if column not in excluded]
inference_features = [
    column for column in inference.columns
    if column not in {"pickup_zone_id", "forecast_hour", "processed_at"}
]
assert train_features == inference_features

# Literal warm-up values are written as `null` in the exports. Fit training-only
# medians and apply the same imputation to validation and inference rows.
train_warmup_nulls = train[train_features].isna().sum().sum()
inference_warmup_nulls = inference[inference_features].isna().sum().sum()
feature_medians = train[train_features].median()
train = train.copy()
inference = inference.copy()
train[train_features] = train[train_features].fillna(feature_medians)
inference[inference_features] = inference[inference_features].fillna(feature_medians)

train_null_counts = train[train_features].isna().sum()
inference_null_counts = inference[inference_features].isna().sum()
assert train_null_counts.sum() == 0
assert inference_null_counts.sum() == 0
assert train["target_pickup_count"].notna().all()
print("Features:", train_features)
print("Training warm-up cells imputed:", int(train_warmup_nulls))
print("Inference warm-up cells imputed:", int(inference_warmup_nulls))


{'zones': 265, 'hourly_fact_rows': 197160, 'training_rows': 197160, 'inference_rows': 197160}
Coverage: 2023-01-01 00:00:00+00:00 to 2023-01-31 23:00:00+00:00
Features: ['hour_of_day', 'day_of_week', 'day_of_month', 'month', 'is_weekend', 'lag_1h', 'lag_24h', 'lag_168h', 'rolling_mean_24h', 'rolling_mean_168h', 'rolling_std_24h']
Training warm-up cells imputed: 52205
Inference warm-up cells imputed: 52205


## 1. Chronological demand forecast

The data is split by time rather than randomly. A lag-1-hour value is the naive
baseline; the boosting model adds calendar, lag, rolling, and variability features.

In [3]:
chronological = train.sort_values(["forecast_hour", "pickup_zone_id"]).reset_index(drop=True)
split_index = int(len(chronological) * 0.80)
split_hour = chronological.iloc[split_index]["forecast_hour"]
train_split = chronological[chronological["forecast_hour"] < split_hour].copy()
test_split = chronological[chronological["forecast_hour"] >= split_hour].copy()

model = HistGradientBoostingRegressor(
    learning_rate=0.08,
    max_iter=200,
    max_leaf_nodes=31,
    l2_regularization=1.0,
    random_state=42,
)
model.fit(train_split[train_features], train_split["target_pickup_count"])
test_split["predicted_pickup_count"] = np.clip(
    model.predict(test_split[train_features]), 0, None
)
test_split["baseline_lag_1h"] = test_split["lag_1h"].clip(lower=0)

def regression_metrics(actual, predicted):
    return {
        "MAE": mean_absolute_error(actual, predicted),
        "RMSE": mean_squared_error(actual, predicted) ** 0.5,
        "R2": r2_score(actual, predicted),
    }

metrics = pd.DataFrame({
    "lag_1h_baseline": regression_metrics(
        test_split["target_pickup_count"], test_split["baseline_lag_1h"]
    ),
    "hist_gradient_boosting": regression_metrics(
        test_split["target_pickup_count"], test_split["predicted_pickup_count"]
    ),
}).T
metrics.round(3)


,MAE,RMSE,R2
lag_1h_baseline,4.169,13.988,0.921
hist_gradient_boosting,2.680,8.620,0.970


In [4]:
predictions = inference[["pickup_zone_id", "forecast_hour"]].copy()
predictions["predicted_pickup_count"] = np.clip(
    model.predict(inference[inference_features]), 0, None
)
predictions = predictions.merge(
    zone, on="pickup_zone_id", how="left", validate="many_to_one"
)
predictions = predictions.merge(
    fact[["pickup_zone_id", "pickup_hour", "pickup_count"]],
    left_on=["pickup_zone_id", "forecast_hour"],
    right_on=["pickup_zone_id", "pickup_hour"],
    how="left",
    validate="one_to_one",
)
predictions["residual"] = (
    predictions["pickup_count"] - predictions["predicted_pickup_count"]
)
predictions.to_csv(data_dir / "demand_predictions.csv", index=False)
joblib.dump(model, data_dir / "demand_hist_gradient_model.joblib")

permutation = permutation_importance(
    model,
    test_split[train_features],
    test_split["target_pickup_count"],
    scoring="neg_mean_absolute_error",
    n_repeats=3,
    random_state=42,
    n_jobs=-1,
)
importance = pd.DataFrame({
    "feature": train_features,
    "importance": permutation["importances_mean"],
}).sort_values("importance", ascending=False)
importance.head(12)


,feature,importance
5,lag_1h,16.282881
7,lag_168h,2.332855
6,lag_24h,2.103178
0,hour_of_day,0.731796
9,rolling_mean_168h,0.531698
8,rolling_mean_24h,0.453187
10,rolling_std_24h,0.246043
1,day_of_week,0.130836
4,is_weekend,0.053447
2,day_of_month,0.000000


## 2. Residual anomaly detection

For each zone, residuals are flagged outside a robust range of median residual plus
or minus three scaled median absolute deviations.

In [5]:
zone_residual_stats = (
    predictions.groupby("pickup_zone_id")["residual"]
    .agg(
        residual_median="median",
        residual_mad=lambda values: np.median(
            np.abs(values - np.median(values))
        ),
    )
    .reset_index()
)
zone_residual_stats["residual_mad"] *= 1.4826

anomalies = predictions.merge(
    zone_residual_stats, on="pickup_zone_id", validate="many_to_one"
)
anomalies["anomaly_deviation"] = (
    (anomalies["residual"] - anomalies["residual_median"]).abs()
    / anomalies["residual_mad"].clip(lower=0.1)
)
anomalies["is_anomaly"] = anomalies["anomaly_deviation"] >= 3.0
anomalies = anomalies[anomalies["is_anomaly"]].copy()
anomalies["anomaly_direction"] = np.where(
    anomalies["residual"] >= 0, "unexpected spike", "unexpected drop"
)
anomalies.to_csv(data_dir / "demand_anomalies.csv", index=False)

print("Anomalies:", len(anomalies))
anomalies.sort_values("anomaly_deviation", ascending=False)[
    [
        "forecast_hour",
        "pickup_zone_id",
        "borough",
        "zone_name",
        "pickup_count",
        "predicted_pickup_count",
        "anomaly_direction",
        "anomaly_deviation",
    ]
].head(15)


Anomalies: 18286


,forecast_hour,pickup_zone_id,borough,zone_name,pickup_count,predicted_pickup_count,anomaly_direction,anomaly_deviation
126480,2023-01-01 00:00:00+00:00,171,Queens,Murray Hill-Queens,0,18.217254,unexpected drop,181.392306
53568,2023-01-01 00:00:00+00:00,73,Queens,East Flushing,0,18.217254,unexpected drop,181.392306
113088,2023-01-01 00:00:00+00:00,153,Manhattan,Marble Hill,0,18.217254,unexpected drop,181.392306
22320,2023-01-01 00:00:00+00:00,31,Bronx,Bronx Park,0,18.217254,unexpected drop,181.392306
14136,2023-01-01 00:00:00+00:00,20,Bronx,Belmont,0,18.217254,unexpected drop,181.392306
7440,2023-01-01 00:00:00+00:00,11,Brooklyn,Bath Beach,0,18.217254,unexpected drop,181.392306
168144,2023-01-01 00:00:00+00:00,227,Brooklyn,Sunset Park East,0,18.217254,unexpected drop,181.392306
156984,2023-01-01 00:00:00+00:00,212,Bronx,Soundview/Bruckner,0,18.217254,unexpected drop,181.392306
74400,2023-01-01 00:00:00+00:00,101,Queens,Glen Oaks,0,18.217254,unexpected drop,181.392306
93000,2023-01-01 00:00:00+00:00,126,Bronx,Hunts Point,0,18.217254,unexpected drop,181.392306


## 3. Zone demand clustering

Each zone is profiled by its hourly pickup pattern, average pickups, variability,
and weekend share. Four standardized K-Means clusters are fitted.

In [6]:
fact_profile = fact.copy()
fact_profile["hour_of_day"] = fact_profile["pickup_hour"].dt.hour
fact_profile["is_weekend"] = fact_profile["pickup_hour"].dt.dayofweek.ge(5).astype(int)

hourly_profile = fact_profile.groupby(
    ["pickup_zone_id", "hour_of_day"]
)["pickup_count"].mean().unstack(fill_value=0)
hourly_profile.columns = [f"hour_{hour}" for hour in hourly_profile.columns]

aggregate_profile = fact_profile.groupby("pickup_zone_id")["pickup_count"].agg(
    avg_pickups="mean",
    std_pickups="std",
    total_pickups="sum",
).reset_index()
weekend_profile = fact_profile.groupby("pickup_zone_id")["is_weekend"].mean().rename(
    "weekend_share"
).reset_index()

profiles = aggregate_profile.merge(
    weekend_profile, on="pickup_zone_id", validate="one_to_one"
)
profiles = profiles.merge(
    hourly_profile.reset_index(), on="pickup_zone_id", validate="one_to_one"
)
profile_features = [column for column in profiles.columns if column != "pickup_zone_id"]
profile_scaled = StandardScaler().fit_transform(
    profiles[profile_features].fillna(0)
)

cluster_model = KMeans(n_clusters=4, n_init=20, random_state=42)
cluster_labels = cluster_model.fit_predict(profile_scaled)
profiles["demand_cluster"] = pd.Categorical(cluster_labels)
profiles = profiles.merge(
    zone, on="pickup_zone_id", how="left", validate="one_to_one"
)
profiles.to_csv(data_dir / "zone_demand_clusters.csv", index=False)
joblib.dump(cluster_model, data_dir / "zone_demand_kmeans.joblib")

cluster_summary = profiles.groupby("demand_cluster", observed=True).agg(
    zones=("pickup_zone_id", "nunique"),
    avg_pickups=("avg_pickups", "mean"),
    std_pickups=("std_pickups", "mean"),
    weekend_share=("weekend_share", "mean"),
).round(3)
cluster_summary


,zones,avg_pickups,std_pickups,weekend_share
demand_cluster,,,,
0,228,1.905,1.816,0.29
1,24,81.119,55.125,0.29
2,5,80.459,76.292,0.29
3,8,167.273,121.224,0.29


## 4. Visual insights

In [7]:
importance = importance.head(12)
fig, ax = plt.subplots(figsize=(10, 5))
sns.barplot(data=importance, x="importance", y="feature", ax=ax, color="#4C78A8")
ax.set_title("Permutation Feature Importance")
ax.set_xlabel("Validation MAE decrease when feature is shuffled")
plt.tight_layout()
plt.show()

sample_test = test_split.sample(min(5000, len(test_split)), random_state=42)
fig, ax = plt.subplots(figsize=(7, 7))
ax.scatter(
    sample_test["target_pickup_count"],
    sample_test["predicted_pickup_count"],
    alpha=0.25,
    s=12,
)
limit = max(
    sample_test["target_pickup_count"].max(),
    sample_test["predicted_pickup_count"].max(),
)
ax.plot([0, limit], [0, limit], color="red", linestyle="--", label="Perfect prediction")
ax.set_xlabel("Actual pickups")
ax.set_ylabel("Predicted pickups")
ax.set_title("Actual vs Predicted Demand")
ax.legend()
plt.tight_layout()
plt.show()

top_anomalies = anomalies.sort_values("anomaly_deviation", ascending=False).head(20)
if len(top_anomalies) == 0:
    print("No residual anomalies exceeded the configured threshold.")
else:
    labels = top_anomalies["zone_name"].fillna(
        top_anomalies["pickup_zone_id"].astype(str)
    )
    fig, ax = plt.subplots(figsize=(10, 7))
    colors = np.where(top_anomalies["residual"] >= 0, "#D62728", "#1F77B4")
    ax.barh(labels.astype(str), top_anomalies["residual"], color=colors)
    ax.set_title("Largest Demand Residual Anomalies")
    ax.set_xlabel("Actual minus predicted pickups")
    ax.invert_yaxis()
    plt.tight_layout()
    plt.show()

hour_columns = [column for column in profiles.columns if column.startswith("hour_")]
hour_means = profiles.groupby("demand_cluster", observed=True)[hour_columns].mean()
hour_means.columns = [int(column.split("_")[1]) for column in hour_means.columns]

fig, ax = plt.subplots(figsize=(12, 6))
for cluster in hour_means.index:
    ax.plot(
        hour_means.columns,
        hour_means.loc[cluster],
        marker="o",
        label=f"Cluster {cluster}",
    )
ax.set_title("Average Hourly Pickup Profile by Zone Cluster")
ax.set_xlabel("Hour of day")
ax.set_ylabel("Average pickups")
ax.set_xticks(range(0, 24))
ax.legend()
plt.tight_layout()
plt.show()


C:\Users\Admin\AppData\Local\Temp\ipykernel_20444\307462272.py:7: UserWarning: FigureCanvasAgg is non-interactive, and thus cannot be shown
  plt.show()
C:\Users\Admin\AppData\Local\Temp\ipykernel_20444\307462272.py:27: UserWarning: FigureCanvasAgg is non-interactive, and thus cannot be shown
  plt.show()
C:\Users\Admin\AppData\Local\Temp\ipykernel_20444\307462272.py:43: UserWarning: FigureCanvasAgg is non-interactive, and thus cannot be shown
  plt.show()
C:\Users\Admin\AppData\Local\Temp\ipykernel_20444\307462272.py:63: UserWarning: FigureCanvasAgg is non-interactive, and thus cannot be shown
  plt.show()


## Generated outputs

- `data/demand_predictions.csv`: all zone-hour forecasts, actual counts, and residuals
- `data/demand_anomalies.csv`: zone-hours classified as demand spikes or drops
- `data/zone_demand_clusters.csv`: four cluster assignments for every zone
- `data/demand_hist_gradient_model.joblib`: trained forecasting model
- `data/zone_demand_kmeans.joblib`: fitted zone clustering model
